In [1]:
# ============================================================================
# scVI RAW COUNT EXPORT  (Notebook 1 / 2)
# Input : per-sample .rds files (Seurat objects)
# Output: per-sample CSV  <SAMPLE>.csv  (genes × barcodes)
#         → upload as a Kaggle dataset, feed into Notebook 2
# ============================================================================

install.packages(c("Seurat", "Matrix"),
                 repos = "https://cloud.r-project.org", quiet = TRUE)

cat("Seurat :", as.character(packageVersion("Seurat")), "\n")
cat("Matrix :", as.character(packageVersion("Matrix")), "\n")


  There is a binary version available but the source version is later:
       binary source needs_compilation
Seurat  5.5.0  5.5.1              TRUE

package 'Matrix' successfully unpacked and MD5 sums checked


installing the source package 'Seurat'




Seurat : 5.5.1 
Matrix : 1.7.5 


In [1]:
library(Seurat)
library(Matrix)

CONFIG <- list(
    rds_root    = "C:/Users/datai/Downloads/ST_Project/dataset/ST",
    output_root = "C:/Users/datai/Downloads/ST_Project/dataset/ST/scVI_counts",
    samples     = c("IU_PDA_HM11", "IU_PDA_HM13", "IU_PDA_T1",
                    "IU_PDA_T11",  "IU_PDA_T3",   "IU_PDA_T4")
)

dir.create(CONFIG$output_root, recursive = TRUE, showWarnings = FALSE)

cat("RDS root   :", CONFIG$rds_root,    "\n")
cat("Output root:", CONFIG$output_root, "\n")
cat("Samples    :", paste(CONFIG$samples, collapse = ", "), "\n")

Loading required package: SeuratObject

Warning message:
"package 'SeuratObject' was built under R version 4.4.3"
Loading required package: sp

Warning message:
"package 'sp' was built under R version 4.4.3"
'SeuratObject' was built with package 'Matrix' 1.7.3 but the current
version is 1.7.5; it is recomended that you reinstall 'SeuratObject' as
the ABI for 'Matrix' may have changed


Attaching package: 'SeuratObject'


The following objects are masked from 'package:base':

    intersect, t


Warning message:
"package 'Matrix' was built under R version 4.4.3"


RDS root   : C:/Users/datai/Downloads/ST_Project/dataset/ST 
Output root: C:/Users/datai/Downloads/ST_Project/dataset/ST/scVI_counts 
Samples    : IU_PDA_HM11, IU_PDA_HM13, IU_PDA_T1, IU_PDA_T11, IU_PDA_T3, IU_PDA_T4 


In [2]:
cat(strrep("=", 70), "\n")
cat("EXPORTING QUALITY-FILTERED RAW COUNTS TO CSV\n")
cat("QC thresholds: nFeature_Spatial >= 200  AND  nCount_Spatial >= 400\n")
cat("  Rationale: HM13 is a lower-depth sample (median nFeature=427, nCount=543).\n")
cat("  Strict thresholds (500/1000) would drop 72% of HM13 spots.\n")
cat("  scVI denoising handles low-count spots; 200/400 removes only background.\n")
cat(strrep("=", 70), "\n\n")

MIN_GENES  <- 200    # minimum unique genes detected per spot
MIN_COUNTS <- 400    # minimum total UMI counts per spot

for (sample_name in CONFIG$samples) {

    out_path <- file.path(CONFIG$output_root, paste0(sample_name, ".csv"))
    qc_path  <- file.path(CONFIG$output_root, paste0(sample_name, "_qc_metrics.csv"))

    # Resume: skip if already exported
    if (file.exists(out_path)) {
        cat("  SKIP (already done):", sample_name, "\n")
        next
    }

    rds_path <- file.path(CONFIG$rds_root, paste0(sample_name, ".rds"))
    if (!file.exists(rds_path)) {
        cat("  WARNING: .rds not found —", rds_path, "\n")
        next
    }

    cat("Loading:", sample_name, "... ")
    obj <- readRDS(rds_path)
    n_raw <- ncol(obj)
    cat(nrow(obj), "genes x", n_raw, "barcodes (raw)\n")

    # ── Export per-spot QC metrics before filtering ───────────────────────
    qc_df <- data.frame(
        barcode  = colnames(obj),
        nFeature = obj@meta.data$nFeature_Spatial,
        nCount   = obj@meta.data$nCount_Spatial,
        stringsAsFactors = FALSE
    )
    qc_df$pass_qc <- (qc_df$nFeature >= MIN_GENES) & (qc_df$nCount >= MIN_COUNTS)
    write.csv(qc_df, file = qc_path, row.names = FALSE)

    # ── Apply spot quality filter ─────────────────────────────────────────
    obj <- subset(obj, subset = nFeature_Spatial >= MIN_GENES &
                                nCount_Spatial   >= MIN_COUNTS)
    n_pass <- ncol(obj)
    n_drop <- n_raw - n_pass
    cat(sprintf("  QC filter: %d/%d spots retained (%d dropped, %.1f%%)\n",
                n_pass, n_raw, n_drop, 100 * n_drop / n_raw))

    # Extract raw counts — handles Seurat v4 and v5
    counts_mat <- tryCatch({
        GetAssayData(obj, assay = "Spatial", slot = "counts")
    }, error = function(e) {
        tryCatch({
            GetAssayData(obj, assay = "Spatial", layer = "counts")
        }, error = function(e2) {
            obj[["Spatial"]]@counts
        })
    })

    cat("  Writing CSV ... ")
    dense_mat <- as.data.frame(as.matrix(counts_mat))
    write.csv(dense_mat, file = out_path, row.names = TRUE)
    cat("done →", out_path, "\n")

    rm(obj, counts_mat, dense_mat); gc()
}

cat("\n", strrep("=", 70), "\n", sep = "")
cat("Export complete — re-run scVI Python notebook on filtered CSVs\n")

EXPORTING QUALITY-FILTERED RAW COUNTS TO CSV
QC thresholds: nFeature_Spatial >= 200  AND  nCount_Spatial >= 400
  Rationale: HM13 is a lower-depth sample (median nFeature=427, nCount=543).
  Strict thresholds (500/1000) would drop 72% of HM13 spots.
  scVI denoising handles low-count spots; 200/400 removes only background.

Loading: IU_PDA_HM11 ... 17893 genes x 3931 barcodes (raw)


Warning message:
"Not validating Seurat objects"


  QC filter: 3894/3931 spots retained (37 dropped, 0.9%)


Warning message:
"The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead."


  Writing CSV ... done → C:/Users/datai/Downloads/ST_Project/dataset/ST/scVI_counts/IU_PDA_HM11.csv 
Loading: IU_PDA_HM13 ... 17893 genes x 2182 barcodes (raw)


...

  QC filter: 2677/2777 spots retained (100 dropped, 3.6%)
  Writing CSV ... done → C:/Users/datai/Downloads/ST_Project/dataset/ST/scVI_counts/IU_PDA_T11.csv 
Loading: IU_PDA_T3 ... 17893 genes x 4354 barcodes (raw)


Warning message:
"Not validating Seurat objects"


  QC filter: 4241/4354 spots retained (113 dropped, 2.6%)
  Writing CSV ... done → C:/Users/datai/Downloads/ST_Project/dataset/ST/scVI_counts/IU_PDA_T3.csv 
Loading: IU_PDA_T4 ... 17893 genes x 3621 barcodes (raw)


Warning message:
"Not validating Seurat objects"


  QC filter: 3587/3621 spots retained (34 dropped, 0.9%)
  Writing CSV ... done → C:/Users/datai/Downloads/ST_Project/dataset/ST/scVI_counts/IU_PDA_T4.csv 

Export complete — re-run scVI Python notebook on filtered CSVs


In [6]:
cat(strrep("=", 70), "\n")
cat("SUMMARY\n")
cat(strrep("=", 70), "\n\n")

for (sample_name in CONFIG$samples) {
    out_path <- file.path(CONFIG$output_root, paste0(sample_name, ".csv"))
    if (file.exists(out_path)) {
        size_mb <- file.info(out_path)$size / 1e6
        cat(sprintf("  ✓ %-20s  %.1f MB\n", sample_name, size_mb))
    } else {
        cat(sprintf("  ✗ %-20s  NOT FOUND\n", sample_name))
    }
}

cat("\nAll files in:", CONFIG$output_root, "\n")
system(paste("ls -lh", CONFIG$output_root))

SUMMARY

  ✓ IU_PDA_HM11           138.1 MB
  ✓ IU_PDA_HM13           22.3 MB
  ✓ IU_PDA_T1             72.4 MB
  ✓ IU_PDA_T11            81.4 MB
  ✓ IU_PDA_T3             141.1 MB
  ✓ IU_PDA_T4             104.5 MB

All files in: C:\Users\datai\Downloads\ST_Project\dataset\ST\ST\scVI_counts 


[1] 0